<a href="https://colab.research.google.com/github/Satyansh-Rawat-131/Violet-An-AI-Ghostwriter/blob/main/VIOLET.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## **TASK**

Our company is not working efficiently we spend tooo much time drafting the documents  and this needs to be fixed

For the company , you need to create an ai agentic ai system that can speed up the drafting documents , emails etc

The ai agentic system should have a Human - AI collaboration meaning the Human should be able to provide cts feedback and the aia agent should be able to stop when thw human is happy with the draft

system should be fast and able to save the drafts

In [ ]:
!pip install -U langchain-openrouter

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.6/90.6 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 837.3/837.3 kB 21.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 463.6/463.6 kB 28.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 57.3 MB/s eta 0:00:00
  Attempting uninstall: pydantic-core
    Found existing installation: pydantic_core 2.46.5
    Uninstalling pydantic_core-2.46.5:
      Successfully uninstalled pydantic_core-2.46.5
  Attempting uninstall: pydantic
    Found existing installation: pydantic 2.13.5
    Uninstalling pydantic-2.13.5:
      Successfully uninstalled pydantic-2.13.5


In [ ]:
from google.colab import userdata
from typing import TypedDict , Annotated , Sequence
from langgraph.graph import START , END , StateGraph
from langchain_openrouter import ChatOpenRouter
from langchain_core.messages import BaseMessage , HumanMessage , AIMessage , ToolMessage , SystemMessage
from langchain_core.tools import tool
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode

In [ ]:
# a global var to store document content
docu_content = ''

In [ ]:
class Agent_State(TypedDict):
  messages : Annotated[Sequence[BaseMessage] , add_messages]

In [ ]:
@tool

def update(content : str) -> str:
  '''Updates the document with the provided content.'''

  global docu_content
  docu_content = content
  return f'Document has been updated successfully! the current content is : \n{docu_content}'

In [ ]:
@tool

def save(filename : str) -> str:
  '''Save the current document to a text file and finish the process.

  ARGS :
    filename : name for the text file.
  '''

  global docu_content

  if not filename.endswith('.txt'): # in case the filename is not saved with .txt extension
    filename = f'{filename}.txt'

  try:
      with open(filename, 'w') as file:
          file.write(docu_content)
      print(f"\n💾 Document has been saved to: {filename}")
      return f"Document has been saved successfully to '{filename}'."

  except Exception as e:
    return f'error in saving the document : {str(e)}'

In [ ]:
# creating lsit of tools

tools = [update , save]

In [ ]:
# calling the model

openrouter_key = userdata.get('API_KEY')
model = ChatOpenRouter(
        model = 'openrouter/free',
        temperature=0.7 ,
        openrouter_api_key = openrouter_key).bind_tools(tools)

In [ ]:
def our_agent(state : Agent_State) -> Agent_State:
  system_prompt = SystemMessage(content = f"""you are a drafter a helpful writting assistant . you are going to help the user update and modify the documents

                                            -if the user wants to update or modify the content  , use the 'update' tool with the complete updated content
                                            -if the user wants to save and finish , you need to use the 'save' tool
                                            -make sure to always show the current document state after modifications

                                            the current document content is : {docu_content}""")
  # an intro message
  if not state['messages']:
    user_input = 'i am ready to help you update a document . what would you like to create'
    user_message = HumanMessage(content = user_input)

  else:
      user_input = input("\nWhat would you like to do with the document? ")
      print(f"\n👤 USER: {user_input}")
      user_message = HumanMessage(content=user_input)


  all_messages = [system_prompt] + list(state['messages']) + [user_message]

  response = model.invoke(all_messages) # invoking the model

  # to make more readable

  print(f"\n🤖 AI: {response.content}")
  if hasattr(response, "tool_calls") and response.tool_calls:
      print(f"🔧 USING TOOLS: {[tc['name'] for tc in response.tool_calls]}")

  return {"messages": list(state["messages"]) + [user_message, response]}

In [ ]:
# creating the conditional edge from tools to agent and back

def should_continue(state : Agent_State) -> str:

  '''determine if we should or end the conversation'''

  messages = state['messages'] # getting the messages and if nothing is there than we will continue and not reach the end path

  if not messages :
    return "continue"

# looks for the most recent tool message

  for message in reversed(messages):
# and checks if this is a toolmessage resulting to save
    if (isinstance(message , ToolMessage) and 'saved' in message.content.lower() and 'document' in message.content.lower()):
      return 'end' # goes to the end which results in the endpoint

  return 'continue'

In [ ]:
def print_messages(messages):
    """Function I made to print the messages in a more readable format"""
    if not messages:
        return

    for message in messages[-3:]:
        if isinstance(message, ToolMessage):
            print(f"\n🛠️ TOOL RESULT: {message.content}")

In [ ]:
graph = StateGraph(Agent_State)

graph.add_node('agent' , our_agent)
graph.add_node('tools' , ToolNode(tools))

graph.set_entry_point('agent')

graph.add_edge('agent' , 'tools')

graph.add_conditional_edges(
    'tools',
    should_continue,
    {
        'continue' : 'agent',
        'end' : END,
    }
)

app = graph.compile()

In [ ]:
def run_document_agent():
    print("\n ===== DRAFTER =====")

    state = {"messages": []}

    for step in app.stream(state, stream_mode="values"):
        if "messages" in step:
            print_messages(step["messages"])

    print("\n ===== DRAFTER FINISHED =====")


In [ ]:
if __name__ == "__main__":
    run_document_agent()


 ===== DRAFTER =====

🤖 AI: I'd be happy to help! To get started, what type of document would you like to create or update?  
Examples:  
- Resume / CV  
- Cover letter  
- Business plan or proposal  
- Essay or article  
- Project plan or meeting notes  
- Something else entirely  

You can tell me the topic, provide any existing draft content, or let me know if you'd like me to generate a template from scratch. Once I have that, I'll use the **update** tool to modify/refine the content, and later the **save** tool to finish and export it.  

What would you like to work on?

What would you like to do with the document? write an email to tom saying i cannot make it to the meeting 

👤 USER: write an email to tom saying i cannot make it to the meeting 

🤖 AI: I'll write a professional email declining the meeting. Let me create that for you:
🔧 USING TOOLS: ['update']

🛠️ TOOL RESULT: Document has been updated successfully! the current content is : 
Subject: Unable to Attend [Meeting Name